# Lab 9: The Age of the Universe
*Elements of Data Science* — adapted from Berkeley Data 8 <br>
Welcome to Lab 9! In Lab 8 you built a **regression toolbox** from scratch: standard units → correlation $r$ → slope and intercept. Now you'll use it on one of the biggest questions in science: **how old is the universe?**

Our estimate comes from a *sample* of exploding stars, so it is subject to sampling error — just like a sample mean. We will measure that uncertainty with the **bootstrap** and report a **confidence interval**. This material is covered in [Chapter 15](https://inferentialthinking.com/chapters/15/Prediction.html) and [Chapter 16](https://inferentialthinking.com/chapters/16/Inference_for_Regression.html) of the online <i>Inferential Thinking</i> textbook.

<br>**<center>Learning Goals**

|Area|Concept|
|---|---|
|Look first|Scatter plot, then a by-eye estimate before any calculation|
|Regression toolbox|`standard_units`, `correlation`, `slope`, `intercept`, `fit_line` from Lab 8|
|Slope with units|Slope = change in $y$ per unit change in $x$; its units are $y$-units ÷ $x$-units|
|Residuals|Error = actual $y$ − predicted $y$; a good fit leaves no pattern|
|$r^2$|Fraction of the variation in $y$ explained by the line|
|Bootstrap|Resample the data **with replacement** to see how much the slope would vary|
|Confidence interval|Middle 95% of bootstrap estimates|

## ⚙️ Initialization — DO NOT EDIT
The next two cells load the course tools (auto-checks, quizzes, styling) and record the lab version. **Do not change them** — with one exception:

👁️ **Accessible view (optional):** set `ACCESSIBLE_VIEW = True` in the cell below for larger, more readable text and colorblind-safe plots. You can switch back any time by setting it to `False` and re-running the cell (then *Kernel → Restart* to clear the text styling).

👉 **Run the cell below first** (Shift+Enter) before anything else in this lab.

In [ ]:
# 🔧 Run this cell first! (Shift+Enter)
ACCESSIBLE_VIEW = True     # True = larger text and colorblind-safe plots

import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "shared-public")))
if ACCESSIBLE_VIEW:
    from init_access import *
else:
    from init import *

In [ ]:
# 🔖 Lab version — do not edit. Used by the automated checks below.
ver = '2026V109'

### 🧰 Standard imports — start your own notebooks with these
The `init` cell above loads the course tools. The four lines below are the **only** imports you need for your own data science notebooks — copy them into the first cell of any new notebook:

```python
import numpy as np                 # arrays, np.arange, np.random.choice, np.mean, np.std ...
from datascience import *          # Table, make_array, are, percentile ...
%matplotlib inline
import matplotlib.pyplot as plt    # plotting
```
Run them here too so every name is defined:

In [ ]:
# Standard imports for this lab
import numpy as np
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plt

Now run the cell below to enter your name.

In [ ]:
# Enter your name as a string
name = ...

## 1. The Actual Big Bang Theory

In the early 20th century, the most popular cosmological theory held that the universe had always existed at a fixed size. Today the **Big Bang** theory prevails: the universe started out very small and is still expanding.

In 1929 Edwin Hubble noticed that distant galaxies are moving **away** from us, and that the farther away a galaxy is, the faster it recedes. This is **Hubble's Law**:

$$ \text{distance} = \text{(time since the start)} \times \text{speed} $$

If every galaxy has been moving away at a steady speed since everything was in one place, then running the motion backwards tells us **when** everything was together — roughly the beginning of the universe. ([Hubble's law](https://en.wikipedia.org/wiki/Hubble%27s_law))

The cell below simulates a universe in which our sun is the center and every other star moves away from it. Each star starts at the same place as the sun; different stars have different directions *and speeds*, shown by the arrows.

**Run the cell, then drag the slider** to move through time.

In [ ]:
# Just run this cell. (Drawing the arrows takes a lot of code; you don't need to read it.)
from ipywidgets import interact, FloatSlider, fixed

num_locations = 20
example_velocities = Table().with_columns(
    "x", np.random.normal(size=num_locations),
    "y", np.random.normal(size=num_locations))
start_of_time = -15

def scatter_after_time(t, start_of_time, end_of_time, velocities, center_name, other_point_name, make_title):
    vx, vy = velocities.column("x"), velocities.column("y")
    max_location = 1.1 * (end_of_time - start_of_time) * max(max(abs(vx)), max(abs(vy)))
    x_now, y_now = (t - start_of_time) * vx, (t - start_of_time) * vy
    plt.figure(figsize=(6, 6))
    plt.scatter(make_array(0), make_array(0), label=center_name, s=250, color='C4', edgecolors='black', zorder=3)
    plt.scatter(x_now, y_now, label=other_point_name, color='C0', zorder=3)
    for i in np.arange(len(vx)):
        plt.arrow(x_now.item(i), y_now.item(i), vx.item(i), vy.item(i),
                  color='black', head_width=0.025 * max_location, lw=0.5)
    plt.xlim(-max_location, max_location)
    plt.ylim(-max_location, max_location)
    plt.gca().set_aspect('equal', adjustable='box')
    plt.legend(loc='upper right')
    plt.title(make_title(t))
    plt.show()

interact(scatter_after_time,
         t=FloatSlider(min=start_of_time, max=5, step=0.5, value=0),
         start_of_time=fixed(start_of_time), end_of_time=fixed(5),
         velocities=fixed(example_velocities),
         center_name=fixed("our sun"), other_point_name=fixed("other star"),
         make_title=fixed(lambda t: "The universe {:g} billion year{} in the {}".format(
             abs(t), "" if abs(t) == 1 else "s", "past" if t < 0 else "future")));

### <font color=blue> **Question 1.1.** </font>
Drag the slider all the way to the left. How many billion years ago did this simulated universe start? Store the number (as a positive number of billion years) in `simulated_start_billion_years_ago`.

In [ ]:
simulated_start_billion_years_ago = ...

In [ ]:
check('tests/q1.1.py')

### <font color=blue> **Question 1.2.** </font>
<font color=green>
1. In your own words, describe Hubble's Law and how it could be used to estimate the age of the universe.
2. After 5 billion years in the future (slider all the way to the right), stars with longer arrows are farther from the sun. Why?
</font>

---

ANSWER

---

#### Save notebook before check

In [ ]:
check('tests/q1.2d.py')

## 2. An Analogy: Drivers Leaving Town

Here's an everyday version of the same idea. Some time in the past, our friend Mei started driving straight east at a steady 60 miles per hour. We are still standing where she started, but we forgot to note the time she left.

If we find out that she is now 120 miles away, we can infer she left **2 hours ago**:

$$ \text{time} = \frac{\text{distance}}{\text{speed}} = \frac{120 \text{ miles}}{60 \text{ miles/hour}} = 2 \text{ hours} $$

Now picture it as a graph with **speed on the horizontal axis** and **distance on the vertical axis**. "Us" (speed 0, distance 0) and Mei (60 mph, 120 miles) are two points. The line through them has slope

$$ \text{slope} = \frac{\text{change in distance}}{\text{change in speed}} = \frac{120 \text{ miles}}{60 \text{ miles/hour}} = 2 \text{ hours} $$

**The slope of distance versus speed is the time since everyone left.** Notice the units: miles ÷ (miles/hour) = hours.

In [ ]:
small_driving_example = Table().with_columns(
    "Name",                                       make_array("Us", "Mei"),
    "Speed moving away from us (miles per hour)", make_array(0,    60),
    "Current distance from us (miles)",           make_array(0,    120))

x = small_driving_example.column("Speed moving away from us (miles per hour)")
y = small_driving_example.column("Current distance from us (miles)")
plt.scatter(x, y, s=200, zorder=3)
plt.plot(x, y)                      # the line through the two points: slope = 120 / 60 = 2 hours
plt.xlabel('Speed moving away from start (miles per hour)')
plt.ylabel('Current distance from start (miles)')
plt.show()

In reality, we never know speeds and distances exactly. But if we measured the speed and distance of **many** drivers who all left at the same time, the slope of a line through all of them would be a good estimate of the time they left — even if each measurement is a little off.

`drivers.csv` contains the speeds and distances of 100 such drivers. **Look first:** run the cell to make a scatter plot.

In [ ]:
drivers = Table.read_table("drivers.csv")
drivers.scatter("Speed moving away from us (miles per hour)", "Current distance from us (miles)")
plt.title('100 drivers who left at the same time')
plt.show()
drivers

<font color=green>**By eye:** a driver going 60 mph is about how many miles away? What slope (and so, what time) does that suggest?</font>

### Your regression toolbox from Lab 8
Rather than guessing, let's **calculate** the slope. Re-create the toolbox you built in Lab 8 below. (Look back at Lab 8 if you need to: [Ch. 15.2](https://inferentialthinking.com/chapters/15/2/Regression_Line.html).)

### <font color=blue> **Question 2.1.** </font>
Fill in the four toolbox functions, then `fit_line`, which returns an array containing the **slope** (item 0) and **intercept** (item 1).

In [ ]:
def standard_units(any_numbers):
    "Convert an array of numbers to standard units."
    return ...

def correlation(t, label_x, label_y):
    "The correlation coefficient r between two columns of Table t."
    return ...

def slope(t, label_x, label_y):
    "Slope of the regression line predicting label_y from label_x, in original units."
    r = correlation(t, label_x, label_y)
    return ...

def intercept(t, label_x, label_y):
    "Intercept of the regression line predicting label_y from label_x, in original units."
    return ...

def fit_line(t, label_x, label_y):
    "Return make_array(slope, intercept) of the regression line."
    return make_array(..., ...)

In [ ]:
# Test: the line through (0, 1) and (1, 3) has slope 2 and intercept 1
example_table = Table().with_columns("x", make_array(0, 1), "y", make_array(1, 3))
fit_line(example_table, "x", "y")

In [ ]:
check('tests/q2.1.py')

### <font color=blue> **Question 2.2.** </font>
`fit_line` returns an **array** of two numbers: item 0 is the slope and item 1 is the intercept. Use `.item()` to pull each one out, as with any array. For the test line in Question 2.1, `fit_line(example_table, "x", "y").item(0)` is 2, the slope.

The first line below fits the regression line to the drivers and stores the array in `driver_line`. Pull out its slope and intercept. The slope is the time since the drivers left, in hours. The cell then draws your line over the data.

In [ ]:
driver_line = fit_line(drivers, "Speed moving away from us (miles per hour)", "Current distance from us (miles)")
driver_slope = ...        # item 0 of driver_line
driver_intercept = ...    # item 1 of driver_line
driving_start_time_hours = driver_slope

speeds = drivers.column("Speed moving away from us (miles per hour)")
plt.scatter(speeds, drivers.column("Current distance from us (miles)"), label='drivers')
xs = make_array(0, 75)
plt.plot(xs, driver_slope * xs + driver_intercept, color='C1', label='regression line')
plt.xlabel('Speed (miles per hour)'); plt.ylabel('Distance (miles)'); plt.legend()
plt.show()
print(f'The drivers left about {driving_start_time_hours:.2f} hours ago.')
print(f'Intercept: {driver_intercept:.2f} miles')

In [ ]:
check('tests/q2.2.py')

<font color=green>**Think about it:** Your intercept is about −1.1 miles. If every driver started at the same place at the same time, what should the intercept be? Why isn't it exactly that?</font>

## 3. Back to Cosmology: Supernovae

To do the same thing for the universe, we need the distance from Earth and the speed away from Earth of many celestial objects. Some of the best sources are **supernovae** — exploding stars that, for a few weeks, shine as brightly as billions of ordinary stars. One kind (Type Ia) explodes with nearly the same brightness every time, so how bright it *looks* tells astronomers how far away it is, and the stretching of its light (redshift) tells them how fast it is moving away.

This Hubble Space Telescope image shows an entire galaxy, with a single supernova — SN 1994D — as the bright spot at the lower left.

<img src="supernova.jpg" width="500" alt="Hubble image of a tilted spiral galaxy with a bright supernova, SN 1994D, at the lower left">

*Image: NASA/ESA, The Hubble Key Project Team and The High-Z Supernova Search Team.*

Our data come from the [Supernova Cosmology Project](https://supernova.lbl.gov/union/) at Lawrence Berkeley National Laboratory. Each row is a supernova near Earth, with its speed (parsecs per year) and distance (millions of parsecs). A **parsec** is a large unit of distance: about 30.86 trillion kilometers, or 3.26 light-years.

In [ ]:
close_novas = Table.read_table("close_novas.csv")
close_novas

### <font color=blue> **Question 3.1.** </font>
**Look first.** Make a scatter plot of the supernovae, with speed on the horizontal axis and distance on the vertical axis. No line yet.

In [ ]:
...

### <font color=blue> **Question 3.2.** </font>
From your scatter plot, estimate the slope by eye: pick a point on the trend and divide its distance by its speed. Then convert to an age in **years**, and store it in `first_guess_universe_age_years`.

⚠️ **Units!** The slope has units of (million parsecs) ÷ (parsecs per year) = **million years**. So a slope of 1 means 1,000,000 years.

In [ ]:
first_guess_universe_age_years = ...   # in years, not billions of years

# Shows your guess in billions of years
print(f"{first_guess_universe_age_years / 1e9:.2f} billion years")

In [ ]:
check('tests/q3.2.py')

## 4. Residuals: How Good Is a Line?

In Lab 8 you met the **error** of a line at each point:

$$ \text{error} = \text{actual distance} - \text{distance predicted by the line} $$

These errors are also called **residuals** — what's left over that the line doesn't explain ([Ch. 15.5](https://inferentialthinking.com/chapters/15/5/Visual_Diagnostics.html)). A **residual plot** shows each residual against $x$. For a good straight-line fit, the residuals should:
- be centered on **0** (positive and negative errors balance out), and
- show **no pattern** — just random scatter. A trend or a curve in the residuals means the line is missing something.

### <font color=blue> **Question 4.1.** </font>
Complete the function `errors`, which returns an array of the residuals for a line with slope `sl` and intercept `inter`. Then compute `example_errors` for the line with slope **16,000** and intercept **0**.

In [ ]:
def errors(t, label_x, label_y, sl, inter):
    """Residuals (actual y minus predicted y) for the line y = sl * x + inter."""
    x = t.column(label_x)
    y = t.column(label_y)
    predicted_y = ...
    return ...

example_errors = errors(close_novas, 'Speed (parsecs/year)', 'Distance (million parsecs)', 16000, 0)
example_errors

In [ ]:
check('tests/q4.1.py')

The cell below draws the line with its residuals underneath on a separate set of axes, so the two never overlap.

In [ ]:
def plot_fit_and_residuals(sl, inter, title=''):
    """Top: data with the line. Bottom: residual plot."""
    x = close_novas.column('Speed (parsecs/year)')
    y = close_novas.column('Distance (million parsecs)')
    res = errors(close_novas, 'Speed (parsecs/year)', 'Distance (million parsecs)', sl, inter)
    sse = np.sum(res**2)
    fig, (top, bottom) = plt.subplots(2, 1, figsize=(8, 7), sharex=True,
                                      gridspec_kw={'height_ratios': [2, 1]})
    top.scatter(x, y, s=15, label='supernovae')
    xl = make_array(0.004, 0.021)
    top.plot(xl, sl * xl + inter, color='C1', label=f'line: slope {sl:,.0f}, intercept {inter:.1f}')
    top.set_ylabel('Distance (million parsecs)'); top.legend(loc='upper left')
    top.set_title(title + f'   SSE = {sse:,.0f}')
    bottom.scatter(x, res, s=15, color='C2')
    bottom.axhline(0, color='black', lw=1)
    bottom.set_ylabel('Residual'); bottom.set_xlabel('Speed (parsecs/year)')
    bottom.set_ylim(-60, 60)
    plt.show()

plot_fit_and_residuals(16000, 0, 'Trial line')
print(f'Average residual: {np.mean(example_errors):.1f} million parsecs')

<font color=green>Almost all of the residuals are negative, and they get more negative as speed increases. What does that tell you about this line — is it too steep or not steep enough?</font>

### Find a better line by eye
Move the sliders to make the SSE as small as you can. Watch the residual plot: you're aiming for points scattered evenly around 0 with no trend.

In [ ]:
interact(plot_fit_and_residuals,
         sl=FloatSlider(min=10000, max=18000, step=100, value=16000, description='slope'),
         inter=FloatSlider(min=-30, max=30, step=1, value=0, description='intercept'),
         title=fixed('Your line'));

### <font color=blue> **Question 4.2.** </font>
Record your best slider slope and intercept, and compute the **mean squared error** of your line:

$$ \text{MSE} = \frac{\sum (y - \hat{y})^2}{n} = \text{mean of the squared residuals} $$

In [ ]:
my_slope = ...
my_intercept = ...
my_errors = errors(close_novas, 'Speed (parsecs/year)', 'Distance (million parsecs)', my_slope, my_intercept)
my_mse = ...
print(f'My line: MSE = {my_mse:.1f}')

In [ ]:
check('tests/q4.2.py')

### <font color=blue> **Question 4.3.** </font>
Now use your toolbox. Fit the regression line to `close_novas` and pull out its slope and intercept, as you did for the drivers in Question 2.2. Then compute:
- `novas_slope` and `novas_intercept`: item 0 and item 1 of `novas_line`
- `novas_r_squared`: the square of the correlation, $r^2$
- `best_errors`: the residuals of the regression line
- `best_mse`: the mean squared error of the regression line
- `age_estimate_billion_years`: the age of the universe, in **billions** of years, from the slope (slope is in millions of years — divide by 1,000)

In [ ]:
novas_line = ...         # fit_line on close_novas: speed (x) and distance (y)
novas_slope = ...        # item 0 of novas_line
novas_intercept = ...    # item 1 of novas_line

novas_r_squared = ...
best_errors = ...
best_mse = ...
age_estimate_billion_years = ...

plot_fit_and_residuals(novas_slope, novas_intercept, 'Regression line')
print(f'Your slider line:  MSE = {my_mse:.1f}')
print(f'Regression line:   MSE = {best_mse:.1f},  r² = {novas_r_squared:.3f}')
print(f'Estimated age of the universe: {age_estimate_billion_years:.2f} billion years')

In [ ]:
check('tests/q4.3.py')

**What does $r^2$ mean?** $r^2$ is the fraction of the variation in distance that is accounted for by the straight-line relationship with speed ([Ch. 15.6](https://inferentialthinking.com/chapters/15/6/Numerical_Diagnostics.html)). An $r^2$ near 1 means the points lie very close to the line; the rest of the variation is left in the residuals.

### <font color=blue> **Question 4.4 Discussion.** </font>
<font color=green>
1. Describe the residual plot of the regression line. Does it have the two qualities of a good fit listed above?
2. Your estimate uses a slope from 156 supernovae, each measured imperfectly. If astronomers had observed a *different* set of 156 supernovae, would you get exactly the same age? How could we find out how much the estimate might change?
</font>

---

ANSWER

---

#### Save notebook before check

In [ ]:
check('tests/q4.4d.py')

### 📝 Quick Check — Slopes, Units, and Residuals
Test what you've learned about Hubble's Law, slope units, and residuals. This is just for practice — answer as many times as you like, it isn't recorded.

In [ ]:
with open("questions09A.json", "r") as f:
    display_quiz(json.load(f), shuffle_answers=True)

## 5. How Uncertain Is Our Age? The Bootstrap

Our supernovae are a **sample**. A different sample would give a slightly different slope, and so a different age. We can't go back to the telescope for new samples — but we can **resample our sample** ([Ch. 13.2](https://inferentialthinking.com/chapters/13/2/Bootstrap.html)):

1. Draw 156 rows from `close_novas` **at random, with replacement**. Some supernovae appear twice or more, others not at all.
2. Fit the regression line to that resample and record the age.
3. Repeat many times.

The spread of the bootstrap ages shows how much our estimate would vary from sample to sample. Applying this to a regression slope is covered in [Ch. 16.2](https://inferentialthinking.com/chapters/16/2/Inference_for_the_True_Slope.html).

`tbl.sample()` with no argument draws a resample of the same size as the table, with replacement. Run the cell below a few times and watch the age change.

In [ ]:
one_resample = close_novas.sample()
one_line = fit_line(one_resample, 'Speed (parsecs/year)', 'Distance (million parsecs)')
print(f'Rows in resample: {one_resample.num_rows}')
print(f'Distinct supernovae in it: {len(np.unique(one_resample.column(0)))}')
print(f'Age from this resample: {one_line.item(0) / 1000:.2f} billion years')

### <font color=blue> **Question 5.1.** </font>
Complete the loop to make **1,000** bootstrap estimates of the age of the universe (in billions of years). Store them in `bootstrap_ages`.

In [ ]:
bootstrap_ages = make_array()
for i in np.arange(...):
    resample = ...
    resample_slope = fit_line(resample, 'Speed (parsecs/year)', 'Distance (million parsecs)').item(0)
    bootstrap_ages = np.append(bootstrap_ages, ...)

len(bootstrap_ages)

In [ ]:
check('tests/q5.1.py')

### <font color=blue> **Question 5.2.** </font>
A **95% confidence interval** is the middle 95% of the bootstrap estimates: from the 2.5th to the 97.5th percentile. The `datascience` function `percentile(p, array)` finds the value at percentile `p`. For example:

In [ ]:
example_values = make_array(10, 20, 30, 40, 50)
percentile(50, example_values)    # the median

Compute `lower_end` and `upper_end` of the 95% confidence interval from `bootstrap_ages`.

In [ ]:
lower_end = ...
upper_end = ...

planck_age = 13.787    # Planck 2018 result, billion years
Table().with_columns("Bootstrap age (billion years)", bootstrap_ages).hist(bins=np.arange(13, 15.2, 0.1))
plt.plot(make_array(lower_end, upper_end), make_array(0, 0), color='C2', lw=10, label='95% confidence interval')
plt.scatter(make_array(planck_age), make_array(0.15), marker='v', s=200, color='C1', edgecolors='black', zorder=3, label='Planck 2018')
plt.legend(loc='upper left')
plt.show()
print(f"95% confidence interval for the age of the universe: [{lower_end:.2f}, {upper_end:.2f}] billion years")

In [ ]:
check('tests/q5.2.py')

### <font color=blue> **Question 5.3 Discussion.** </font>
<font color=green>
1. What is your 95% confidence interval for the age of the universe? The [Planck satellite (2018)](https://arxiv.org/abs/1807.06209) measured the age as 13.787 ± 0.020 billion years using the cosmic microwave background. Is that value inside your interval?
2. Does your confidence interval for the **slope** include 0? What would it mean if it did? Connect this to the null hypothesis "there is no relationship between speed and distance" from Lab 7.
3. Our supernovae weren't chosen at random — astronomers measured the ones easiest to see. How might that affect our estimate or our confidence interval?
4. In your own words, how did the bootstrap estimate the uncertainty without collecting any new data?
</font>

---

ANSWER

---

#### Save notebook before check

In [ ]:
check('tests/q5.3d.py')

### 👁️ Optional: plots with alt text
If you turned on `ACCESSIBLE_VIEW` at the top, you can display any figure with a written description (**alt text**) that screen readers announce. Writing a good one-sentence description is also a great way to check that *you* understand your plot. Try it:

In [ ]:
if ACCESSIBLE_VIEW:
    plt.scatter(close_novas.column(0), close_novas.column(1), s=15)
    plt.xlabel('Speed (parsecs/year)'); plt.ylabel('Distance (million parsecs)')
    show_plot("Scatter plot of 156 supernovae: distance rises in a nearly straight line as speed increases.")
else:
    print("Set ACCESSIBLE_VIEW = True in the first cell and re-run it to try show_plot().")

## 📝 Quick Review Quiz
Unlike the quick check above, this quiz **is recorded** — you get two attempts per question, so take your time.

In [ ]:
# Interactive quiz — run this cell and answer the questions!
from stemds_quiz import open_quiz
quiz_result = open_quiz("questions09B.json", name, user, quiz_id="Lab09B", max_attempts=2)

## 6. Reflection

### <font color=blue> **Question 6.** </font>
At the end of each lab, please include a reflection.
* How did this lab go?
* Did using your regression toolbox from Lab 8 make this lab easier?
* Did you try the accessible view? If so, was it helpful?
* Were there questions you found especially challenging that you would like your instructor to review in class?
* How long did the lab take you to complete?

Share your feedback so we can continue to improve this class!

**Use the markdown cell below this one and write your reflection on this lab.** Nice work, data astronomer!

---

ANSWER

---

#### Save notebook before check

In [ ]:
check('tests/q6d.py')

## 🏁 Final Step — Check, Submit to Canvas, Then Submit Lab

Follow these three steps **in order**:

1. ✅ **Run the cell below** to test all your answers and see your score.
2. 📤 **Upload to Canvas.** Download your notebook as a `.ipynb` file ([instructions](https://laserchemist.github.io/data/submission-instructions.html#ipynb)) and export an HTML copy ([instructions](https://laserchemist.github.io/data/submission-instructions.html#html)), then upload that HTML file to this lab's Canvas assignment.
3. 🚀 **Only then run the final cell and click Submit Lab.** This records your score to the course gradebook — treat it as confirmation that steps 1 and 2 are already done.

In [ ]:
# Run all tests at once
from gofer.ok import check
correct = 0
questions = ['1.1', '1.2d',
             '2.1', '2.2',
             '3.2',
             '4.1', '4.2', '4.3', '4.4d',
             '5.1', '5.2', '5.3d',
             '6d']
for x in questions:
    print("Testing question {}: ".format(x))
    display(check("tests/q{}.py".format(x)))
    score = check('tests/q{}.py'.format(str(x)))
    if score.grade == 1.0:
        correct += 1

In [ ]:
from lab_submit import submit_lab
submit_lab(name, user, correct, questions, lab_id="Lab09")